In [1]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import os
import re

import torch
import torch.nn as nn
from scipy import signal
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split, KFold, GroupKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,  balanced_accuracy_score, f1_score
import numpy as np
from itertools import combinations
from scipy.signal import butter, filtfilt


In [2]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # For full determinism (slower but stable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Load data and manual labels

In [3]:
# Load manual sleep stage classification csv
CSV_PATH = Path("eeg_epochs_dataset.csv")
DATA_ROOT = Path("current_study_data_raw/") 
df = pd.read_csv(CSV_PATH)
# Sleep stage mapping
LABEL_MAP = {"W": 0, "N1": 1}

df = df[df["stage_label"].isin(LABEL_MAP.keys())].copy()
df["y"] = df["stage_label"].map(LABEL_MAP).astype(int)

df["epoch_start_time"] = pd.to_numeric(df["epoch_start_time"], errors="coerce")
df["epoch_end_time"]   = pd.to_numeric(df["epoch_end_time"], errors="coerce")
df["y"]                = pd.to_numeric(df["y"], errors="coerce")

df = df.dropna(subset=["epoch_start_time", "epoch_end_time", "y"])

df["subject_id"] = pd.to_numeric(df["subject_id"], errors="coerce")
df = df.dropna(subset=["subject_id"])
df["subject_id"] = df["subject_id"].astype(int)


In [4]:
subject_list = []
for folder in os.listdir(DATA_ROOT):
    m = re.search(r"\d+", folder)
    if m:
        subject_list.append(int(m.group()))
subject_set = set(subject_list)

In [5]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="No coordinate information found for channels*",
    category=RuntimeWarning,
)
warnings.filterwarnings(
    "ignore",
    message="Not setting positions of*misc channels found in montage*",
    category=RuntimeWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Not setting positions of 4 misc channels found in montage*",
    category=RuntimeWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Not setting positions of 6 misc channels found in montage*",
    category=RuntimeWarning,
)

## Create datasets

In [16]:
def load_raw_brainvision(subject_id: int) -> mne.io.BaseRaw:
    subdir = DATA_ROOT / f"H{subject_id:03d}"
    candidates = [
        f"{subject_id:03d}_scan.vhdr",
        f"H{subject_id:03d}_scan.vhdr",
        f"sub-{subject_id:03d}.vhdr",
        "scanner.vhdr",
        "scan.vhdr",
    ]

    for fname in candidates:
        vhdr_path = subdir / fname
        if vhdr_path.exists():
            return mne.io.read_raw_brainvision(
                vhdr_path, preload=False, verbose=False
            )
    raise FileNotFoundError(f"No .vhdr found for subject {subject_id} in {subdir}")

X_list, y_list, subject_id_list = [], [], []

TARGET_CHANNELS = ["EOG1"]

for subject_id, sdf in df.groupby("subject_id"):
    subject_id = int(subject_id)
    if subject_id not in subject_set:
        continue
    print("Loading subject", subject_id)
    raw = load_raw_brainvision(subject_id)

    if not all(ch in raw.ch_names for ch in TARGET_CHANNELS):
        print(f"Skip subject {subject_id}: missing channels")
        continue

    raw.pick_channels(TARGET_CHANNELS, ordered=True)
    raw.resample(256)

    sfreq = raw.info["sfreq"]
    n_samp = int(round(30.0 * sfreq))
    n_total = raw.n_times

    for row in sdf.itertuples(index=False):
        start_samp = int(float(row.epoch_start_time) * sfreq)
        if start_samp < 0 or start_samp + n_samp > n_total:
            continue

        seg = raw.get_data(start=start_samp, stop=start_samp + n_samp)
        X_list.append(seg)
        y_list.append(int(row.y))
        subject_id_list.append(subject_id)



Loading subject 6
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 12
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 16
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 17
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 19
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 22
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 23
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 24
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 26
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading subject 28
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading sub

In [17]:
print("epochs:", len(X_list), "labels:", len(y_list))
print("example seg shape:", X_list[0].shape)
print("label counts:", np.bincount(np.array(y_list)))


epochs: 610 labels: 610
example seg shape: (1, 7680)
label counts: [419 191]


In [18]:
X = np.array(X_list)
y=np.array(y_list)

In [19]:
X.shape

(610, 1, 7680)

In [20]:
y.shape

(610,)

In [21]:
# 1. Prepare the 'groups' array (must match the length of X and y)
# You'll need to create this during your data loading loop
# For example: groups = np.array(subject_id_list)
groups = np.array(subject_id_list) # Assuming you tracked these in X_list loop
kf = GroupKFold(n_splits=5)

# One-time: compute subject-wise folds once and reuse everywhere
splits = list(kf.split(np.zeros(len(y)), y, groups=groups))

# Optional sanity check: print test subjects per fold
for f, (_, va) in enumerate(splits):
    print(f"Fold {f+1} test subjects:", np.unique(groups[va]))

Fold 1 test subjects: [  6  28  46  51  61  63 100 120]
Fold 2 test subjects: [ 12  16  17  22  53  65  93 116]
Fold 3 test subjects: [ 19  23  26  42  55  57  69  97 115]
Fold 4 test subjects: [ 24  32  36  47  48  91 109 117]
Fold 5 test subjects: [ 34  38  43  49  67  95 107 112 119]


## Random Forest approach

In [22]:
def extract_eeg_features(epoch, fs=256):
    """
    Extract time-domain and frequency-domain features from a single epoch
    
    Args:
        epoch: shape (3, 7680) - 3 channels
        fs: sampling frequency in Hz (adjust based on your actual sampling rate)
    
    Returns:
        features: 1D array of extracted features
    """
    features = []
    theta_sum = 0.0
    alpha_sum = 0.0
    
    # Process each channel
    for channel in range(epoch.shape[0]):
        signal_data = epoch[channel, :]
        
        # === TIME DOMAIN FEATURES ===
        # Statistical features
        features.append(np.mean(signal_data))
        features.append(np.std(signal_data))
        features.append(np.var(signal_data))
        features.append(skew(signal_data))
        features.append(kurtosis(signal_data))
        features.append(np.max(signal_data))
        features.append(np.min(signal_data))
        features.append(np.ptp(signal_data))  # peak-to-peak
        features.append(np.median(signal_data))
        
        # Root mean square
        features.append(np.sqrt(np.mean(signal_data**2)))
        
        # Zero crossing rate
        zero_crossings = np.sum(np.diff(np.sign(signal_data)) != 0)
        features.append(zero_crossings)
        
        # === FREQUENCY DOMAIN FEATURES ===
        # Compute power spectral density
        freqs, psd = signal.welch(signal_data, fs=fs, nperseg=min(256, len(signal_data)))
        
        # Define frequency bands (Hz)
        # Adjust these based on your sampling rate and sleep stage research
        delta_band = (0.5, 4)    # Deep sleep
        theta_band = (4, 8)      # Light sleep, drowsiness
        alpha_band = (8, 13)     # Relaxed wakefulness
        beta_band = (13, 30)     # Active thinking, alert
        gamma_band = (30, 50)    # High-level information processing
        
        bands = [delta_band, theta_band, alpha_band, beta_band, gamma_band]
        
        for low, high in bands:
            # Find indices for this frequency band
            idx = np.logical_and(freqs >= low, freqs <= high)
            
            # Band power (absolute)
            band_power = np.trapezoid(psd[idx], freqs[idx])
            features.append(band_power)
        
        # Total power
        total_power = np.trapezoid(psd, freqs)
        features.append(total_power)
        
        # Relative band powers (percentage of total power)
        for low, high in bands:
            idx = np.logical_and(freqs >= low, freqs <= high)
            band_power = np.trapezoid(psd[idx], freqs[idx])
            relative_power = band_power / (total_power + 1e-10)  # avoid division by zero
            features.append(relative_power)
        
        # # === CROSS-CHANNEL FEATURE (1 feature only) ===
        # # Mean pairwise theta-band correlation (4–8 Hz)

        # # Bandpass filter design
        # nyq = 0.5 * fs
        # low = 4 / nyq
        # high = 8 / nyq
        # b, a = signal.butter(4, [low, high], btype='band')

        # # Bandpass all channels
        # theta_signals = []
        # for channel in range(epoch.shape[0]):
        #     filtered = signal.filtfilt(b, a, epoch[channel, :])
        #     theta_signals.append(filtered)

        # # Compute mean pairwise correlation
        # corr_vals = []
        # for i in range(len(theta_signals)):
        #     for j in range(i + 1, len(theta_signals)):
        #         xi = theta_signals[i]
        #         xj = theta_signals[j]
        #         if np.std(xi) < 1e-12 or np.std(xj) < 1e-12:
        #             corr_vals.append(0.0)
        #         else:
        #             corr_vals.append(np.corrcoef(xi, xj)[0, 1])

        # mean_theta_corr = np.mean(corr_vals) if len(corr_vals) > 0 else 0.0
        # features.append(mean_theta_corr)
        
    return np.array(features)


# Extract features for all epochs
print("Extracting features from all epochs...")
X_features = []

for i in range(X.shape[0]):
    epoch_features = extract_eeg_features(X[i], fs=256)  # Adjust fs if needed
    X_features.append(epoch_features)
    
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{X.shape[0]} epochs")

X_features = np.array(X_features)
print(f"Feature matrix shape: {X_features.shape}")

fold_acc = []
fold_bacc = []
fold_f1 = []

for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n--- RF Fold {fold+1}/5 (Test subjects: {np.unique(groups[val_idx])}) ---")

    X_tr, X_va = X_features[train_idx], X_features[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,   # fixed like your CNN seeding idea
        n_jobs=-1
    )

    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_va)

    acc = accuracy_score(y_va, y_pred)
    bacc = balanced_accuracy_score(y_va, y_pred)
    f1 = f1_score(y_va, y_pred)   # for binary (pos label=1)

    fold_acc.append(acc)
    fold_bacc.append(bacc)
    fold_f1.append(f1)

    print(f"Accuracy: {acc:.4f}") #| Balanced Acc: {bacc:.4f} | F1: {f1:.4f}")
    print(classification_report(y_va, y_pred, target_names=['Class 0', 'Class 1']))
    print(confusion_matrix(y_va, y_pred))

print("\nRF Subject-wise CV:")
print(f"Accuracy:       {np.mean(fold_acc):.4f} (+/- {np.std(fold_acc):.4f})")
print(f"Balanced Acc:   {np.mean(fold_bacc):.4f} (+/- {np.std(fold_bacc):.4f})")
print(f"F1 (class=1):   {np.mean(fold_f1):.4f} (+/- {np.std(fold_f1):.4f})")

Extracting features from all epochs...
Processed 100/610 epochs
Processed 200/610 epochs
Processed 300/610 epochs
Processed 400/610 epochs
Processed 500/610 epochs
Processed 600/610 epochs
Feature matrix shape: (610, 22)

--- RF Fold 1/5 (Test subjects: [  6  28  46  51  61  63 100 120]) ---
Accuracy: 0.6303
              precision    recall  f1-score   support

     Class 0       0.68      0.88      0.76        81
     Class 1       0.29      0.11      0.15        38

    accuracy                           0.63       119
   macro avg       0.48      0.49      0.46       119
weighted avg       0.55      0.63      0.57       119

[[71 10]
 [34  4]]

--- RF Fold 2/5 (Test subjects: [ 12  16  17  22  53  65  93 116]) ---
Accuracy: 0.7250
              precision    recall  f1-score   support

     Class 0       0.75      0.94      0.83        88
     Class 1       0.44      0.12      0.20        32

    accuracy                           0.72       120
   macro avg       0.60      0.53    

In [13]:
from sklearn.metrics import cohen_kappa_score
cohen_kappa_score(y_va, y_pred)

-0.002865329512893977

## CNN approach

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_channels = X.shape[1]  

# Normalization
X_norm = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-8)

fold_accuracies = []
fold_bacc = []
fold_f1 = []

epochs = 100 
batch_size = 64

# Convert data to tensors
X_tensor = torch.FloatTensor(X_norm).to(device) 
y_tensor = torch.LongTensor(y).to(device)

# 2. Split using groups=groups
for fold, (train_idx, val_idx) in enumerate(splits):
    seed = 42 + fold  # different but deterministic per fold
    set_seed(seed)
    print(f"\n--- Starting Fold {fold + 1}/5 (Testing on Subject IDs: {np.unique(groups[val_idx])}) ---")
    
    # Split data
    train_X, val_X = X_tensor[train_idx], X_tensor[val_idx]
    train_y, val_y = y_tensor[train_idx], y_tensor[val_idx]
    
    # Re-initialize Model for a fresh start
    model = nn.Sequential(
        nn.Conv1d(input_channels, 32, kernel_size=50, stride=6),
        nn.ReLU(),
        nn.MaxPool1d(8),
        nn.Conv1d(32, 64, kernel_size=8),
        nn.ReLU(),
        nn.MaxPool1d(8),
        nn.Flatten(),
        nn.LazyLinear(128),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128, 2)
    ).to(device)
    
    # Handle class imbalance (419 vs 191)
    counts = torch.bincount(train_y)
    weights = (counts.sum() / (2.0 * counts.float())).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    # Training
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(len(train_X), device=train_X.device)

        
        for i in range(0, len(train_X), batch_size):
            idx = perm[i:i+batch_size]
            batch_X = train_X[idx]
            batch_y = train_y[idx]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
    # Evaluation
    model.eval()
    with torch.no_grad():
        val_outputs = model(val_X)
        val_preds = torch.argmax(val_outputs, dim=1)
        
        # Calculate accuracy for this fold
        correct = (val_preds == val_y).sum().item()
        #accuracy = correct / len(val_y)
        accuracy = accuracy_score(val_y.cpu().numpy(), val_preds.cpu().numpy())
        bacc = balanced_accuracy_score(val_y.cpu().numpy(), val_preds.cpu().numpy())
        f1 = f1_score(val_y.cpu().numpy(), val_preds.cpu().numpy())   # for binary (pos label=1)
        
        fold_accuracies.append(accuracy)
        fold_bacc.append(bacc)
        fold_f1.append(f1)
        
        print(f"Fold {fold + 1} Subject-Wise Accuracy: {accuracy:.4f} | Balanced Acc: {bacc:.4f} | F1: {f1:.4f}")
        print("\nClassification Report:")
        print(classification_report(val_y.cpu().numpy(), val_preds.cpu().numpy(), target_names=['Class 0', 'Class 1']))
        print("\nConfusion Matrix:")
        print(confusion_matrix(val_y.cpu().numpy(), val_preds.cpu().numpy()))

print(f"\nFinal Subject-Wise CV Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")
print(f"Balanced Accuracy: {np.mean(fold_bacc):.4f} (+/- {np.std(fold_bacc):.4f})")
print(f"F1 Score: {np.mean(fold_f1):.4f} (+/- {np.std(fold_f1):.4f})")


--- Starting Fold 1/5 (Testing on Subject IDs: [  6  28  46  51  61  63 100 120]) ---
Fold 1 Subject-Wise Accuracy: 0.6303 | Balanced Acc: 0.5328 | F1: 0.3125

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.70      0.80      0.75        81
     Class 1       0.38      0.26      0.31        38

    accuracy                           0.63       119
   macro avg       0.54      0.53      0.53       119
weighted avg       0.60      0.63      0.61       119


Confusion Matrix:
[[65 16]
 [28 10]]

--- Starting Fold 2/5 (Testing on Subject IDs: [ 12  16  17  22  53  65  93 116]) ---
Fold 2 Subject-Wise Accuracy: 0.6500 | Balanced Acc: 0.5227 | F1: 0.2759

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.74      0.80      0.77        88
     Class 1       0.31      0.25      0.28        32

    accuracy                           0.65       120
   macro avg       0.53      0.52      0.52      

In [15]:
from sklearn.metrics import cohen_kappa_score
cohen_kappa_score(val_y, val_preds)

-0.024390243902439046

## Transfer learning with SleepXViT (pretrained model)

This section is specifically for **SleepXViT** (`sarahhyojin/SleepXViT`) fine-tuning on your current `X`, `y`, and subject-wise `splits`.

Before running, follow the SleepXViT repo setup in your environment (clone repo + install requirements + download pretrained checkpoint). Then set `SLEEPXVIT_REPO` and `PRETRAINED_WEIGHTS` below.


In [ ]:
# SleepXViT transfer learning on your current dataset tensors (X, y, splits)
from pathlib import Path
import sys
import copy
import importlib
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report

# --------------------
# 1) SleepXViT setup
# --------------------
SLEEPXVIT_REPO = Path("external/SleepXViT")
PRETRAINED_WEIGHTS = Path("external/SleepXViT/checkpoints/pretrained_sleepxvit.pth")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training config
BATCH_SIZE = 64
EPOCHS_FROZEN = 5
EPOCHS_FINETUNE = 15
LR_HEAD = 1e-3
LR_ALL = 1e-4
WEIGHT_DECAY = 1e-4
NUM_CLASSES = len(np.unique(y))


def import_sleepxvit_model_class(repo_path: Path):
    """Try common SleepXViT module paths from the official repo."""
    if not repo_path.exists():
        raise FileNotFoundError(
            f"SleepXViT repo not found at {repo_path}. "
            "Clone it first, e.g. `git clone https://github.com/sarahhyojin/SleepXViT external/SleepXViT`."
        )

    if str(repo_path.resolve()) not in sys.path:
        sys.path.insert(0, str(repo_path.resolve()))

    candidates = [
        ("models.sleepxvit", "SleepXViT"),
        ("model.sleepxvit", "SleepXViT"),
        ("sleepxvit", "SleepXViT"),
    ]

    for module_name, class_name in candidates:
        try:
            mod = importlib.import_module(module_name)
            if hasattr(mod, class_name):
                return getattr(mod, class_name)
        except Exception:
            pass

    raise ImportError(
        "Could not import SleepXViT class automatically. "
        "Open the SleepXViT repo and update `candidates` with the correct module/class path."
    )


def replace_classifier_head(model: nn.Module, out_classes: int):
    """Replace common classifier head names for transfer learning."""
    head_attr_names = ["head", "classifier", "fc", "mlp_head"]

    for attr in head_attr_names:
        if hasattr(model, attr):
            old_head = getattr(model, attr)
            if isinstance(old_head, nn.Linear):
                in_features = old_head.in_features
                setattr(model, attr, nn.Linear(in_features, out_classes))
                return [attr]
            if isinstance(old_head, nn.Sequential) and len(old_head) > 0 and isinstance(old_head[-1], nn.Linear):
                in_features = old_head[-1].in_features
                new_head = nn.Sequential(*list(old_head[:-1]), nn.Linear(in_features, out_classes))
                setattr(model, attr, new_head)
                return [attr]

    raise AttributeError(
        "Could not find a supported classifier head (head/classifier/fc/mlp_head). "
        "Inspect SleepXViT model and update `replace_classifier_head`."
    )


def freeze_backbone_except(model: nn.Module, trainable_attr_names):
    for p in model.parameters():
        p.requires_grad = False
    for attr in trainable_attr_names:
        for p in getattr(model, attr).parameters():
            p.requires_grad = True


# --------------------
# 2) Build model + load pretrained checkpoint
# --------------------
SleepXViTClass = import_sleepxvit_model_class(SLEEPXVIT_REPO)

# NOTE: constructor signature may differ across versions.
# If needed, modify these kwargs to match SleepXViT's README/config.
model = SleepXViTClass(num_classes=NUM_CLASSES)
head_names = replace_classifier_head(model, NUM_CLASSES)

if not PRETRAINED_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Pretrained checkpoint not found at {PRETRAINED_WEIGHTS}. "
        "Download the official pretrained SleepXViT weights and set PRETRAINED_WEIGHTS correctly."
    )

checkpoint = torch.load(PRETRAINED_WEIGHTS, map_location="cpu")
state = checkpoint.get("state_dict", checkpoint.get("model_state_dict", checkpoint))

# strip common DataParallel prefix if present
state = {k.replace("module.", "", 1): v for k, v in state.items()}

# Load all compatible pretrained weights (classifier mismatches are expected)
missing, unexpected = model.load_state_dict(state, strict=False)
print("Loaded SleepXViT pretrained weights")
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model = model.to(DEVICE)


# --------------------
# 3) Data prep (reuse your existing X/y/splits)
# --------------------
X_norm = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-8)

counts = np.bincount(y)
cls_weights = counts.sum() / (len(counts) * counts)
class_weights = torch.tensor(cls_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)


def evaluate(net, loader):
    net.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = net(xb)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            y_pred.append(pred)
            y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    return (
        balanced_accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        y_true,
        y_pred,
    )


def train_one_stage(net, loader, optimizer):
    net.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = net(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()


# --------------------
# 4) GroupKFold transfer learning
# --------------------
fold_metrics = []
all_true, all_pred = [], []

for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
    set_seed(42 + fold)

    # re-init per fold
    fold_model = copy.deepcopy(model)

    X_tr = torch.tensor(X_norm[tr_idx], dtype=torch.float32)
    y_tr = torch.tensor(y[tr_idx], dtype=torch.long)
    X_va = torch.tensor(X_norm[va_idx], dtype=torch.float32)
    y_va = torch.tensor(y[va_idx], dtype=torch.long)

    tr_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)

    # Stage A: train head only
    freeze_backbone_except(fold_model, head_names)
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, fold_model.parameters()), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    best_bacc = -1.0
    best_state = None

    for _ in range(EPOCHS_FROZEN):
        train_one_stage(fold_model, tr_loader, opt)
        bacc, _, _, _ = evaluate(fold_model, va_loader)
        if bacc > best_bacc:
            best_bacc = bacc
            best_state = copy.deepcopy(fold_model.state_dict())

    # Stage B: unfreeze all and fine-tune
    for p in fold_model.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(fold_model.parameters(), lr=LR_ALL, weight_decay=WEIGHT_DECAY)

    for _ in range(EPOCHS_FINETUNE):
        train_one_stage(fold_model, tr_loader, opt)
        bacc, _, _, _ = evaluate(fold_model, va_loader)
        if bacc > best_bacc:
            best_bacc = bacc
            best_state = copy.deepcopy(fold_model.state_dict())

    fold_model.load_state_dict(best_state)
    bacc, f1m, y_t, y_p = evaluate(fold_model, va_loader)

    fold_metrics.append((bacc, f1m))
    all_true.append(y_t)
    all_pred.append(y_p)
    print(f"Fold {fold}: bAcc={bacc:.4f}, F1-macro={f1m:.4f}")

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)
print("\nSleepXViT transfer learning summary")
print("Mean bAcc:", np.mean([m[0] for m in fold_metrics]))
print("Mean F1-macro:", np.mean([m[1] for m in fold_metrics]))
print("\nClassification report:\n", classification_report(all_true, all_pred, digits=4))

